<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson03/hints_1_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hints 1: Your own image dataset

Hints for Assignment 02, in three parts:

1. **Build your own image dataset** ← *you are here*
2. Fine-tune a model on it
3. Turn it into a web app

**The goal:** 2-5 categories, 100-300 images per category, about 1,000 images in total, split into `train` and `test` folders (75% : 25%).

Pick categories you find fun, but not trivially easy to tell apart. Here we use three kinds of bears as an example.

You can also take the photos yourself, e.g. with your phone! Then skip section 2, put your photos into one folder per category inside a folder `raw`, and continue with section 3.

A CPU runtime is fine for this notebook, you don't need a GPU.

In [ ]:
!pip install -q ddgs

In [ ]:
import random
import shutil

from ddgs import DDGS
from fastai.vision.all import *

---
## 1. Your categories

In [ ]:
categories = ["grizzly bear", "black bear", "teddy bear"]   # TODO: your own 2-5 categories
dataset_name = "bears"                                      # TODO: the name of your dataset

---
## 2. Search and download images

We search for images with the [ddgs](https://pypi.org/project/ddgs/) library, which asks web search engines (DuckDuckGo, Bing, ...). One search returns a page of image URLs:

In [ ]:
with DDGS() as ddgs:
    results = ddgs.images("teddy bear", max_results=100)

print(len(results), "results")
results[0]      # look inside: one result

We only need the `"image"` URL of each result. One page has only a few dozen images, so this function asks for more pages until it has enough URLs:

In [ ]:
def search_images(keyword, max_images=300):
    urls = []
    page = 1
    with DDGS() as ddgs:
        while len(urls) < max_images:
            results = ddgs.images(keyword, max_results=100, page=page)
            new_urls = [result["image"] for result in results if result["image"] not in urls]
            if not new_urls:
                break           # no more results
            urls += new_urls
            page += 1
    return urls[:max_images]

Now download the images, each category into its own folder inside `raw`. fastai's `download_images` downloads many images at once. Some downloads fail, that's fine. This takes a minute or two per category:

In [ ]:
raw = Path("raw")
raw.mkdir(exist_ok=True)

for category in categories:
    urls = search_images(f"{category} photo")
    download_images(raw / category, urls=urls)
    print(f"{category}: {len(urls)} URLs, {len(get_image_files(raw / category))} images downloaded")

A search engine runs out of new results after 100-200 images. If you need more, search again with a different keyword and download into the same folder, for example:

```python
download_images(raw / "grizzly bear", urls=search_images("grizzly bear in the wild"))
```

---
## 3. Clean up

Some downloaded files are broken or are not images at all, and the images come in different formats (`.png`, `.webp`, ...) and sizes. This helper turns every image into a `.jpg` of at most 400×400 pixels, and deletes the files that cannot be opened:

In [ ]:
def to_small_jpg(file, max_size=400):
    try:
        image = Image.open(file).convert("RGB")
    except Exception:
        file.unlink()                          # not an image -> delete it
        return
    image.thumbnail((max_size, max_size))      # shrink, keep the aspect ratio
    image.save(file.with_suffix(".jpg"), "JPEG")
    if file.suffix != ".jpg":
        file.unlink()                          # delete the original .png, .webp, ...

for file in list(raw.rglob("*")):
    if file.is_file():
        to_small_jpg(file)

### Look inside: your images

The first 8 images of each category:

In [ ]:
for category in categories:
    image_files = get_image_files(raw / category)
    print(f"{category}: {len(image_files)} images")
    show_images([PILImage.create(f) for f in image_files[:8]], nrows=1, figsize=(16, 2))
    plt.show()

**Look at your images!** A web search also returns wrong, duplicate or useless pictures (drawings, logos, collages...). Browse the folders (folder icon on the left, double-click an image to open it) and delete the worst ones. Don't worry about every single image: in part 2, the trained model will help you find the suspicious ones.

---
## 4. Split into train and test (75% : 25%)

We shuffle the images of each category and copy a quarter of them into `test`, the rest into `train`:

In [ ]:
dataset = Path(dataset_name)
random.seed(42)

for category in categories:
    image_files = list(get_image_files(raw / category))
    random.shuffle(image_files)
    n_test = len(image_files) // 4
    for i, file in enumerate(image_files):
        split = "test" if i < n_test else "train"
        (dataset / split / category).mkdir(parents=True, exist_ok=True)
        shutil.copy(file, dataset / split / category / file.name)

### Look inside: the folder structure

This is the required format (see the assignment): `train` and `test`, and inside each one folder per category:

In [ ]:
for split in ["train", "test"]:
    for category in categories:
        print(f"{dataset_name}/{split}/{category}/: {len(get_image_files(dataset / split / category))} images")

---
## 5. Save it and make it public

Colab deletes all files when the runtime disconnects, so zip the dataset and download it to your laptop:

In [ ]:
!zip -qr {dataset_name}.zip {dataset_name}

from google.colab import files
files.download(f"{dataset_name}.zip")

Now make the dataset publicly available, for example:
- upload the ZIP to your Google Drive and share it (*Share → General access → Anyone with the link*), or
- create a [HuggingFace dataset](https://huggingface.co/new-dataset) and upload the ZIP there.

You will submit the link in the form. Keep the ZIP, you will need it in [part 2](https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson03/hints_2_finetuning.ipynb).